In [13]:
import json
import folium
from shapely.geometry import Point, Polygon, LineString, shape, MultiPolygon
from shapely.ops import unary_union
from pyproj import Transformer

# Load the Pontaise limits GeoJSON
with open('delimites.geojson', 'r') as f:
    pontaise_limits = json.load(f)

# Load the numeros merged GeoJSON
with open('1888-cadastre-renove/src/numeros_merged-v3.geojson', 'r') as f:
    numeros_data = json.load(f)

# Extract all geometries from the GeoJSON and convert to polygons
polygons = []
for feature in pontaise_limits['features']:
    geom = shape(feature['geometry'])
    if isinstance(geom, LineString):
        # Convert LineString to Polygon by closing it
        poly = Polygon(geom.coords)
        polygons.append(poly)
        print(f"Converted LineString with {len(geom.coords)} vertices to Polygon")
    elif isinstance(geom, Polygon):
        polygons.append(geom)
        print(f"Found Polygon with {len(geom.exterior.coords)} vertices")

# Create a union of all polygons to handle multiple areas
if len(polygons) > 1:
    pontaise_polygon = unary_union(polygons)
    print(f"\nCreated union of {len(polygons)} polygons")
else:
    pontaise_polygon = polygons[0]
    print(f"\nUsing single polygon")

# Create a transformer to convert from WGS84 (lon/lat) to EPSG:2056 (Swiss coordinates)
# The boundary is in EPSG:2056, and the points are in WGS84/CRS84
transformer = Transformer.from_crs("EPSG:4326", "EPSG:2056", always_xy=True)

print("Transforming points from WGS84 to EPSG:2056...")

# Filter points that are inside the Pontaise boundary (with coordinate transformation)
filtered_features = []
for feature in numeros_data['features']:
    lon, lat = feature['geometry']['coordinates']
    # Transform to EPSG:2056 to match the boundary coordinate system
    x, y = transformer.transform(lon, lat)
    point = Point(x, y)
    
    if pontaise_polygon.contains(point):
        filtered_features.append(feature)

print(f"Filtered {len(filtered_features)} points inside Pontaise boundary")

# Create new GeoJSON with filtered points
pontaise_points = {
    "type": "FeatureCollection",
    "name": "pontaise_points",
    "crs": numeros_data.get('crs'),
    "features": filtered_features
}

# Save to new GeoJSON file
with open('pontaise_points.geojson', 'w') as f:
    json.dump(pontaise_points, f, indent=2)

print(f"Saved to pontaise_points.geojson")

# Create Folium map for visualization (using WGS84 coordinates)
# Transform the boundary polygon back to WGS84 for display
transformer_back = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)

# Convert boundary polygon(s) to WGS84
def transform_polygon_to_wgs84(poly):
    """Transform a polygon from EPSG:2056 to WGS84"""
    exterior_coords = []
    for x, y in poly.exterior.coords:
        lon, lat = transformer_back.transform(x, y)
        exterior_coords.append([lon, lat])
    return exterior_coords

# Handle both Polygon and MultiPolygon
boundary_coords_list = []
if isinstance(pontaise_polygon, MultiPolygon):
    for poly in pontaise_polygon.geoms:
        boundary_coords_list.append(transform_polygon_to_wgs84(poly))
else:
    boundary_coords_list.append(transform_polygon_to_wgs84(pontaise_polygon))

# Calculate center for map
if filtered_features:
    # Use first point as reference for centering
    sample_coords = filtered_features[0]['geometry']['coordinates']
    center_lon, center_lat = sample_coords[0], sample_coords[1]
else:
    center_lat, center_lon = 46.528, 6.630

# Create map
m = folium.Map(location=[center_lat, center_lon], zoom_start=15)

# Add the Pontaise boundary polygons
for i, boundary_coords in enumerate(boundary_coords_list):
    folium.Polygon(
        locations=[[lat, lon] for lon, lat in boundary_coords],
        color='blue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.1,
        weight=3,
        popup=f'Pontaise Boundary {i+1}'
    ).add_to(m)

# Add the filtered points
for feature in filtered_features:
    coords = feature['geometry']['coordinates']
    properties = feature['properties']
    
    # Create popup text with properties
    popup_text = "<br>".join([f"<b>{k}:</b> {v}" for k, v in properties.items()])
    
    folium.CircleMarker(
        location=[coords[1], coords[0]],  # lat, lon
        radius=3,
        popup=folium.Popup(popup_text, max_width=300),
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.7
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

print(f"Map created with {len(filtered_features)} points")
print(f"Boundary contains {len(boundary_coords_list)} polygon(s)")

# Display the map
m

Found Polygon with 6 vertices
Found Polygon with 11 vertices

Created union of 2 polygons
Transforming points from WGS84 to EPSG:2056...
Filtered 733 points inside Pontaise boundary
Saved to pontaise_points.geojson
Map created with 733 points
Boundary contains 2 polygon(s)
Saved to pontaise_points.geojson
Map created with 733 points
Boundary contains 2 polygon(s)


In [10]:
# Debug: Check coordinate ranges
print("=== Boundary Polygon Info ===")
print(f"Polygon type: {type(pontaise_polygon)}")
print(f"Polygon bounds: {pontaise_polygon.bounds}")  # (minx, miny, maxx, maxy)
print(f"Polygon centroid: {pontaise_polygon.centroid}")

print("\n=== Points Info ===")
if numeros_data['features']:
    first_point = numeros_data['features'][0]['geometry']['coordinates']
    print(f"First point coordinates: {first_point}")
    print(f"Total points in dataset: {len(numeros_data['features'])}")
    
    # Check a few sample points
    print("\nSample of first 5 points:")
    for i, feature in enumerate(numeros_data['features'][:5]):
        coords = feature['geometry']['coordinates']
        point = Point(coords)
        inside = pontaise_polygon.contains(point)
        print(f"  Point {i}: {coords} - Inside: {inside}")
        
# Check CRS
print("\n=== CRS Info ===")
print(f"Boundary CRS: {pontaise_limits.get('crs', 'Not specified')}")
print(f"Points CRS: {numeros_data.get('crs', 'Not specified')}")

=== Boundary Polygon Info ===
Polygon type: <class 'shapely.geometry.multipolygon.MultiPolygon'>
Polygon bounds: (2536336.8196544745, 1152870.1330306903, 2538258.581425641, 1153797.718390779)
Polygon centroid: POINT (2537440.224758 1153255.632045492)

=== Points Info ===
First point coordinates: [6.638899715922851, 46.526967274330254]
Total points in dataset: 13887

Sample of first 5 points:
  Point 0: [6.638899715922851, 46.526967274330254] - Inside: False
  Point 1: [6.64001008605638, 46.51893189032875] - Inside: False
  Point 2: [6.636989656636587, 46.516921750768496] - Inside: False
  Point 3: [6.633068190046516, 46.52296420714562] - Inside: False
  Point 4: [6.631963409215524, 46.52221543049124] - Inside: False

=== CRS Info ===
Boundary CRS: {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:EPSG::2056'}}
Points CRS: {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}


In [11]:
from pyproj import Transformer

# Create a transformer to convert from WGS84 (lon/lat) to EPSG:2056 (Swiss coordinates)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:2056", always_xy=True)

print("Transforming points from WGS84 to EPSG:2056...")

# Filter points that are inside the Pontaise boundary (with coordinate transformation)
filtered_features_corrected = []
for feature in numeros_data['features']:
    lon, lat = feature['geometry']['coordinates']
    # Transform to EPSG:2056
    x, y = transformer.transform(lon, lat)
    point = Point(x, y)
    
    if pontaise_polygon.contains(point):
        filtered_features_corrected.append(feature)

print(f"Filtered {len(filtered_features_corrected)} points inside Pontaise boundary")

# Create new GeoJSON with filtered points
pontaise_points_corrected = {
    "type": "FeatureCollection",
    "name": "pontaise_points",
    "crs": numeros_data.get('crs'),
    "features": filtered_features_corrected
}

# Save to new GeoJSON file
with open('pontaise_points.geojson', 'w') as f:
    json.dump(pontaise_points_corrected, f, indent=2)

print(f"Saved to pontaise_points.geojson")

Transforming points from WGS84 to EPSG:2056...
Filtered 733 points inside Pontaise boundary
Saved to pontaise_points.geojsonSaved to pontaise_points.geojson



In [12]:
# Create Folium map for visualization (using WGS84 coordinates)
# Transform the boundary polygon back to WGS84 for display
transformer_back = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)

# Convert boundary polygon(s) to WGS84
def transform_polygon_to_wgs84(poly):
    """Transform a polygon from EPSG:2056 to WGS84"""
    exterior_coords = []
    for x, y in poly.exterior.coords:
        lon, lat = transformer_back.transform(x, y)
        exterior_coords.append([lon, lat])
    return exterior_coords

# Handle both Polygon and MultiPolygon
boundary_coords_list = []
if isinstance(pontaise_polygon, MultiPolygon):
    for poly in pontaise_polygon.geoms:
        boundary_coords_list.append(transform_polygon_to_wgs84(poly))
else:
    boundary_coords_list.append(transform_polygon_to_wgs84(pontaise_polygon))

# Calculate center for map
if filtered_features_corrected:
    # Use first point as reference for centering
    sample_coords = filtered_features_corrected[0]['geometry']['coordinates']
    center_lon, center_lat = sample_coords[0], sample_coords[1]
else:
    center_lat, center_lon = 46.528, 6.630

# Create map
m = folium.Map(location=[center_lat, center_lon], zoom_start=15)

# Add the Pontaise boundary polygons
for i, boundary_coords in enumerate(boundary_coords_list):
    folium.Polygon(
        locations=[[lat, lon] for lon, lat in boundary_coords],
        color='blue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.1,
        weight=3,
        popup=f'Pontaise Boundary {i+1}'
    ).add_to(m)

# Add the filtered points
for feature in filtered_features_corrected:
    coords = feature['geometry']['coordinates']
    properties = feature['properties']
    
    # Create popup text with properties
    popup_text = "<br>".join([f"<b>{k}:</b> {v}" for k, v in properties.items()])
    
    folium.CircleMarker(
        location=[coords[1], coords[0]],  # lat, lon
        radius=3,
        popup=folium.Popup(popup_text, max_width=300),
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.7
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

print(f"Map created with {len(filtered_features_corrected)} points")
print(f"Boundary contains {len(boundary_coords_list)} polygon(s)")

# Display the map
m

Map created with 733 points
Boundary contains 2 polygon(s)
